# Phase 1 — Eval Harness Calibration

Three experiments, in order:

1. **Dataset audit** — is the golden set structurally sound?
2. **No-retrieval baseline** — what does the bare LLM already know?
3. **Judge calibration** — does the LLM judge agree with human labels?

Conclusions graduate to `README.md` in this directory, then to ADRs.
This notebook is evidence, not documentation — it may be discarded once
conclusions are recorded.

In [ ]:
import json
from pathlib import Path
from collections import Counter

REPO_ROOT = Path.cwd().parent.parent
GOLDEN_PATH = REPO_ROOT / "data" / "golden_dataset.jsonl"
RUBRIC_PATH = REPO_ROOT / "doc" / "judge-rubric.md"

golden = [json.loads(line) for line in GOLDEN_PATH.read_text().splitlines() if line.strip()]
rubric = RUBRIC_PATH.read_text()
print(f"{len(golden)} golden questions loaded, rubric {len(rubric)} chars")

## 1. Dataset audit

Checks: all six types covered, no `TODO` evidence left in non-seed entries,
answerable questions have quoted evidence, unanswerable/out-of-domain have none.

In [ ]:
print(Counter(q["type"] for q in golden))

seeds = [q["id"] for q in golden if q.get("status") == "seed_example"]
todos = [q["id"] for q in golden if "TODO" in json.dumps(q) and q.get("status") != "seed_example"]
print(f"seed examples still present: {seeds}")
print(f"non-seed entries with TODOs (must be empty): {todos}")

## 2. No-retrieval baseline

Bare LLM, zero context. For each answerable question: does the model already
know the answer from training data?

**What to look for:** if the model nails a large share of these, the dataset
is skewed toward famous papers and questions must be rewritten toward
recent/obscure work — otherwise retrieval can never prove its value.

In [ ]:
# TODO: wire the LLM client (decision pending in doc/project-status.md)
# def bare_llm_answer(question: str) -> str: ...
#
# baseline = [{"id": q["id"], "answer": bare_llm_answer(q["question"])} 
#             for q in golden if q["expected_behavior"] == "answer"]
# Path("baseline_answers.json").write_text(json.dumps(baseline, indent=2))

## 3. Human labels

Hand-score ~20 (question, answer) pairs against the rubric BEFORE running
the judge. These labels are the ground truth the judge is calibrated against.
Store as `human_labels.json`: `[{"id", "verdict", "reason"}]`.

## 4. Judge run + disagreement analysis

Run the judge (rubric as prompt) on the same pairs. The deliverable is the
disagreement table — every row where judge != human forces either a rubric
fix (bump version in doc/judge-rubric.md, re-run) or a label fix.

In [ ]:
# TODO after sections 2-3:
# judge_verdicts = [judge(q, answer, evidence, rubric) for ...]
# disagreements = [pair for pair in zip(human, judge_verdicts) if verdicts differ]
# agreement = 1 - len(disagreements) / len(human)
# Iterate until agreement >= 0.90, bumping rubric version each change.

## 5. Conclusions

> Fill in, then copy to README.md and open ADRs.

- Judge–human agreement: `___%` at rubric `v_`
- No-retrieval baseline accuracy: `___%` (n=`__` answerable questions)
- Questions rewritten as too-famous: `___`
- Rubric changes made and why: `___`